# Notebook 02v2: Collect Hidden States (Bug-Fixed)

Changes from v1:
- `enable_thinking=False` unified across all files
- `num_train=500, num_eval=500` (paper scale)
- Validates Single Answer MATH accuracy as sanity check

In [ ]:
import os, sys, subprocess
REPO = 'thoughtcomm'
if os.path.exists(REPO):
    os.chdir(REPO)
    subprocess.run(['git', 'pull', 'origin', 'main'], check=True)
else:
    subprocess.run(['git', 'clone', 'https://github.com/AUMEZAK/thoughtcomm.git'], check=True)
    os.chdir(REPO)
    subprocess.run(['pip', 'install', '-e', '.', '-q'], check=True)
# Verify v2 fixes
exec(open('tests/test_v2_fixes.py').read())

In [ ]:
# Mount Google Drive for checkpoints
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import torch
from configs.config import ThoughtCommConfig
from models.model_utils import load_model_and_tokenizer
from data.math_data import load_math_dataset
from data.gsm8k_data import load_gsm8k_dataset
from pipeline.debate import MultiAgentDebate
from pipeline.hidden_state_collector import HiddenStateCollector
import os

config = ThoughtCommConfig.for_qwen_0_6b()
print(f'Model: {config.model_name}')
print(f'num_train={config.num_train}, num_eval={config.num_eval}')
print(f'n_h={config.n_h} ({config.num_agents} agents x {config.hidden_size})')

MODEL_TAG = config.model_name.replace('/', '_')
SAVE_DIR = config.save_dir
os.makedirs(SAVE_DIR, exist_ok=True)

In [ ]:
# Load model
model, tokenizer = load_model_and_tokenizer(
    config.model_name, dtype=config.torch_dtype
)
print(f'Model loaded: {config.model_name}')
print(f'Pad token: {tokenizer.pad_token} (id={tokenizer.pad_token_id})')

In [ ]:
# Load datasets (paper scale: 500+500)
math_train, math_eval = load_math_dataset(
    num_train=config.num_train, num_eval=config.num_eval, level=config.math_level
)
gsm8k_train, gsm8k_eval = load_gsm8k_dataset(
    num_train=config.num_train, num_eval=config.num_eval
)
print(f'MATH: {len(math_train)} train, {len(math_eval)} eval')
print(f'GSM8K: {len(gsm8k_train)} train, {len(gsm8k_eval)} eval')

In [ ]:
# Sanity check: Single Answer on 10 MATH problems
# This verifies enable_thinking=False is working
from pipeline.thought_comm import run_single_answer_baseline
from evaluation.math_eval import extract_boxed_answer, grade_answer

sa_sample = run_single_answer_baseline(model, tokenizer, math_eval[:10], config)
correct = 0
for resp, ex in zip(sa_sample, math_eval[:10]):
    pred = extract_boxed_answer(resp)
    if pred and grade_answer(pred, ex['answer']):
        correct += 1
print(f'Single Answer sanity check: {correct}/10 correct ({correct*10}%)')
print('Expected ~40-50% if enable_thinking fix is working')
print('If ~0-5%, enable_thinking fix is NOT applied')

# Show a sample response to verify no <think> tags
print(f'\nSample response (first 200 chars):\n{sa_sample[0][:200]}')

In [ ]:
# Collect hidden states for MATH training set
collector = HiddenStateCollector(model, tokenizer, config)
H_math, meta_math = collector.collect(
    math_train, save_dir=os.path.join(SAVE_DIR, f'{MODEL_TAG}_math_v2')
)
print(f'MATH hidden states: {H_math.shape}')

In [ ]:
# Collect hidden states for GSM8K training set
H_gsm8k, meta_gsm8k = collector.collect(
    gsm8k_train, save_dir=os.path.join(SAVE_DIR, f'{MODEL_TAG}_gsm8k_v2')
)
print(f'GSM8K hidden states: {H_gsm8k.shape}')

In [ ]:
# Save combined hidden states
import json

torch.save({'H': H_math, 'metadata': meta_math},
           os.path.join(SAVE_DIR, f'{MODEL_TAG}_math_hidden_v2.pt'))
torch.save({'H': H_gsm8k, 'metadata': meta_gsm8k},
           os.path.join(SAVE_DIR, f'{MODEL_TAG}_gsm8k_hidden_v2.pt'))

# Verify
summary = {
    'model': config.model_name,
    'math_h_shape': list(H_math.shape),
    'math_h_mean': H_math.float().mean().item(),
    'math_h_std': H_math.float().std().item(),
    'math_num_samples': len(meta_math),
    'gsm8k_h_shape': list(H_gsm8k.shape),
    'gsm8k_h_mean': H_gsm8k.float().mean().item(),
    'gsm8k_h_std': H_gsm8k.float().std().item(),
    'gsm8k_num_samples': len(meta_gsm8k),
}

os.makedirs('results', exist_ok=True)
with open(f'results/02v2_hidden_states_{MODEL_TAG}.json', 'w') as f:
    json.dump(summary, f, indent=2)

for k, v in summary.items():
    print(f'{k}: {v}')